# Agent Memory & Cognitive Architectures

**Level:** Advanced · **Time:** 90 min

In this notebook, we simulate the mechanics of a multi-layered memory system, moving beyond the "dump it all in a vector DB" anti-pattern.

We will cover 4 distinct patterns:
1. **The Infinite Context Anti-Pattern:** Why just appending messages eventually crashes the agent.
2. **Memory Consolidation:** Extracting concrete facts from noisy chat logs.
3. **Contradiction Resolution:** Superseding old facts to maintain a single source of truth.
4. **Memory Isolation (RAG Leaks):** Why vector search without hard filtering leaks data across tenants.

---
## Pattern 1: The Infinite Context Anti-Pattern

A naive agent just appends every turn to its prompt. Eventually, it hits the token limit or suffers from 'Lost in the Middle'.

In [1]:
class NaiveAgent:
    def __init__(self):
        self.context_window = []
        self.token_limit = 50
        
    def chat(self, message):
        print(f"[User] {message}")
        self.context_window.append(message)
        
        # Simulating token exhaustion
        total_tokens = sum(len(m.split()) for m in self.context_window)
        if total_tokens > self.token_limit:
            print(f"🚨 FATAL: Context window exhausted ({total_tokens}/{self.token_limit} tokens). Agent crashed.")
            return False
            
        print(f"  [Agent] Acknowledged. Context size: {total_tokens} tokens.")
        return True

agent = NaiveAgent()
agent.chat("Hi, my name is Alice.")
agent.chat("I really like Python.")
agent.chat("Can you tell me about the weather?")
agent.chat("Here is a very long log file that goes on and on and on and on and on and on and on and on and on...")
agent.chat("What was my name again?")


[User] Hi, my name is Alice.
  [Agent] Acknowledged. Context size: 5 tokens.
[User] I really like Python.
  [Agent] Acknowledged. Context size: 9 tokens.
[User] Can you tell me about the weather?
  [Agent] Acknowledged. Context size: 16 tokens.
[User] Here is a very long log file that goes on and on and on and on and on and on and on and on and on...
  [Agent] Acknowledged. Context size: 42 tokens.
[User] What was my name again?
  [Agent] Acknowledged. Context size: 47 tokens.


True

---
## Pattern 2: Memory Consolidation (Reflection)

Instead of keeping raw logs, a background process extracts concrete semantic facts and deletes the raw logs.

In [2]:
semantic_db = {}

def reflection_engine(raw_chat_log):
    print("\n[Reflection Engine] Analyzing episodic logs...")
    # Simulated LLM extraction logic
    if "like Python" in raw_chat_log:
        fact = {"topic": "favorite_language", "value": "Python"}
        semantic_db["favorite_language"] = fact
        print(f"✅ [Reflection] Extracted Fact: {fact}")
        print("🗑️ [Reflection] Expiring raw episodic logs to save space.")

raw_logs = "User: I really like Python. Agent: That's great."
reflection_engine(raw_logs)
print(f"Semantic DB State: {semantic_db}")



[Reflection Engine] Analyzing episodic logs...
✅ [Reflection] Extracted Fact: {'topic': 'favorite_language', 'value': 'Python'}
🗑️ [Reflection] Expiring raw episodic logs to save space.
Semantic DB State: {'favorite_language': {'topic': 'favorite_language', 'value': 'Python'}}


---
## Pattern 3: Contradiction Resolution

If the user changes their mind, we don't store both facts. We supersede the old fact to maintain a Single Source of Truth.

In [3]:
def update_semantic_memory(topic, new_value):
    print(f"\n[DB] Upserting fact for '{topic}': {new_value}")
    
    if topic in semantic_db:
        print(f"⚠️ [DB] Contradiction detected. Superseding old value '{semantic_db[topic]['value']}'.")
    
    semantic_db[topic] = {"topic": topic, "value": new_value}

update_semantic_memory("favorite_language", "Go")
print(f"Semantic DB State: {semantic_db}")

print("\n[Agent Query] What is the user's favorite language?")
print(f"[Agent Answer] It is {semantic_db['favorite_language']['value']}. (No LLM guessing required)")



[DB] Upserting fact for 'favorite_language': Go
⚠️ [DB] Contradiction detected. Superseding old value 'Python'.
Semantic DB State: {'favorite_language': {'topic': 'favorite_language', 'value': 'Go'}}

[Agent Query] What is the user's favorite language?
[Agent Answer] It is Go. (No LLM guessing required)


---
## Pattern 4: Memory Isolation (The RAG Leak)

If you rely purely on vector similarity, you will leak data across tenants. You must use Hybrid Retrieval (Hard filter + Vector).

In [4]:
mock_vector_db = [
    {"tenant": "Acme", "content": "Acme secret formula is X."},
    {"tenant": "Globex", "content": "Globex secret formula is Y."}
]

def naive_vector_search(query):
    print(f"\n[Naive Search] Searching for '{query}'...")
    # Simulating the vector DB finding the closest semantic match regardless of tenant
    print(f"🚨 DATA LEAK: Returned '{mock_vector_db[0]['content']}' to Globex user!")

def hybrid_retrieval(tenant_id, query):
    print(f"\n[Hybrid Search] Pre-filtering for tenant_id '{tenant_id}'...")
    isolated_pool = [doc for doc in mock_vector_db if doc["tenant"] == tenant_id]
    
    # Simulating vector search on the isolated pool
    print(f"✅ SECURE: Returned '{isolated_pool[0]['content']}' to Globex user.")

naive_vector_search("secret formula")
hybrid_retrieval(tenant_id="Globex", query="secret formula")



[Naive Search] Searching for 'secret formula'...
🚨 DATA LEAK: Returned 'Acme secret formula is X.' to Globex user!

[Hybrid Search] Pre-filtering for tenant_id 'Globex'...
✅ SECURE: Returned 'Globex secret formula is Y.' to Globex user.
